# Portugal day-ahead prices — complete 2024 extraction and basic statistics

This notebook retrieves **all 12 calendar months of 2024** for the Portugal (PT) bidding zone from a secondary, OMIE-derived public archive. It validates completeness before reporting annual statistics. The source's 15-minute arrays for 2024 usually repeat each hourly price four times; this notebook collapses them to hourly prices. The archive is **not a verified official OMIE API**. Other products (IDA, continuous, aFRR, mFRR, imbalance) are **not** included in this day-ahead analysis.

Run cells top to bottom in an internet-enabled Python environment. Do **not** interpret the unexecuted notebook as a completed annual dataset.

In [ ]:
import io, json, time
from pathlib import Path
import numpy as np
import pandas as pd
import requests
import matplotlib.pyplot as plt

YEAR = 2024
BASE = 'https://raw.githubusercontent.com/tiagofelicia/dados-energia/main/data/mapas/precos_qh'
OUT = Path('data')
OUT.mkdir(exist_ok=True)
session = requests.Session()

In [ ]:
def fetch_month(month):
    url = f'{BASE}/{YEAR}-{month:02d}.json'
    last_exc = None
    for attempt in range(4):
        try:
            response = session.get(url, timeout=75)
            response.raise_for_status()
            payload = response.json()
            if not isinstance(payload, dict) or not payload:
                raise ValueError('Empty or invalid JSON')
            print(f'{month:02d}: received {len(payload)} days')
            return payload
        except (requests.RequestException, ValueError) as exc:
            last_exc = exc
            time.sleep(min(2**attempt, 8))
    raise RuntimeError(f'Failed to fetch {url}: {last_exc}')

months = {}
failed = {}
for m in range(1, 13):
    try:
        months[m] = fetch_month(m)
    except Exception as exc:
        failed[m] = str(exc)
print('Months fetched:', len(months), '/ 12')
if failed:
    print('FAILED MONTHS:', failed)
    print('Annual analysis must not be interpreted as complete.')

In [ ]:
records = []
issues = []
for m, payload in months.items():
    for day, zones in payload.items():
        pt = zones.get('PT') if isinstance(zones, dict) else None
        if not isinstance(pt, dict) or not isinstance(pt.get('values'), list):
            issues.append((day, 'missing PT values'))
            continue
        values = pt['values']
        if len(values) in (92, 96, 100):
            arr = np.asarray(values, dtype=float).reshape(-1, 4)
            if not np.all(np.isclose(arr, arr[:, :1], atol=1e-6)):
                issues.append((day, 'quarter-hour values not identical; cannot assume hourly price'))
                continue
            prices = arr[:, 0]
        elif len(values) in (23, 24, 25):
            prices = np.asarray(values, dtype=float)
        else:
            issues.append((day, f'unexpected {len(values)} prices'))
            continue
        if not np.isfinite(prices).all():
            issues.append((day, 'nonfinite price'))
            continue
        for i, price in enumerate(prices):
            records.append({'date': day, 'hour_ordinal': i + 1, 'price_eur_mwh': float(price)})
df = pd.DataFrame.from_records(records, columns=['date','hour_ordinal','price_eur_mwh'])
if not df.empty:
    df['date'] = pd.to_datetime(df['date'])
    df = df.sort_values(['date', 'hour_ordinal']).reset_index(drop=True)
print('Extracted rows:', len(df), 'Days:', df['date'].nunique() if len(df) else 0)
print('Parsing issues:', issues[:25], '; total:', len(issues))

In [ ]:
expected_days = pd.date_range('2024-01-01', '2024-12-31', freq='D')
observed_days = pd.DatetimeIndex(df['date'].unique()) if not df.empty else pd.DatetimeIndex([])
missing_days = expected_days.difference(observed_days)
extra_days = observed_days.difference(expected_days)
by_day = df.groupby('date').size() if not df.empty else pd.Series(dtype='int64')
invalid_day_lengths = by_day[~by_day.isin([23, 24, 25])]
complete = (not failed and not issues and len(missing_days)==0 and len(extra_days)==0 and invalid_day_lengths.empty and len(observed_days)==366)
print('Complete:', complete)
print('Missing days:', [str(x.date()) for x in missing_days])
print('Extra days:', [str(x.date()) for x in extra_days])
print('Invalid day lengths:', invalid_day_lengths.to_dict())
print('2024 expected calendar days: 366; DST may produce 23- or 25-hour local days.')
if not complete:
    print('WARNING: do not label statistics below as full-year 2024.')

In [ ]:
output_csv = OUT / 'portugal_day_ahead_2024_hourly.csv'
df.to_csv(output_csv, index=False)
print('Wrote:', output_csv, 'rows:', len(df))
if complete:
    print('FULL YEAR VALIDATED')
else:
    print('PARTIAL DATASET ONLY: missing months/days or validation issues')

In [ ]:
if df.empty:
    raise RuntimeError('No valid observations available')
series = df['price_eur_mwh']
summary = pd.Series({'observations': int(series.count()), 'days': int(df['date'].nunique()), 'mean': series.mean(), 'median': series.median(), 'std': series.std(), 'min': series.min(), 'p05': series.quantile(.05), 'p25': series.quantile(.25), 'p75': series.quantile(.75), 'p95': series.quantile(.95), 'max': series.max(), 'negative_hours': int((series<0).sum()), 'zero_hours': int((series==0).sum())}, name='value')
print('Dataset status:', 'COMPLETE 2024' if complete else 'PARTIAL, NOT ANNUAL')
print(summary.to_string())
monthly = df.assign(month=df.date.dt.to_period('M')).groupby('month').price_eur_mwh.agg(['count','mean','median','std','min','max'])
print(monthly)
if complete:
    summary.to_csv(OUT / 'portugal_day_ahead_2024_summary.csv', header=True)
    monthly.to_csv(OUT / 'portugal_day_ahead_2024_monthly.csv')

In [ ]:
daily = df.groupby('date').price_eur_mwh.mean()
fig, axes = plt.subplots(2, 1, figsize=(12, 8), constrained_layout=True)
daily.plot(ax=axes[0], title='PT day-ahead daily average — 2024' + ('' if complete else ' (PARTIAL DATA)'))
axes[0].set_ylabel('EUR/MWh')
monthly['mean'].plot.bar(ax=axes[1], title='PT day-ahead monthly average' + ('' if complete else ' (PARTIAL DATA)'))
axes[1].set_ylabel('EUR/MWh')
plt.show()

## Interpretation and data limitations

- Statistics represent **day-ahead energy prices**, not retail bills or ancillary-service clearing prices.
- The source is a republished archive, so results should be reconciled with official OMIE annual/monthly figures before being called official.
- Hour ordinals are sequential within each day. They avoid inventing ambiguous local timestamps at daylight-saving transitions.
- Full-year claims are valid only if the `complete` flag above is `True`; otherwise the results describe **only the successfully extracted subset**.
- Next analyses: hourly profiles, weekday/weekend comparisons, seasonality, volatility, negative-price frequency and comparisons with 2025.